# Phase 14: Full ImageNet Validation

This notebook validates the single best Phase 13 setting at larger scale.

Primary claim test:

- Untouched released dSVA checkpoint on the exact same ImageNet validation split.
- Matched DINO+MAE continuation as a diagnostic only.
- DINO+MAE+I-JEPA continuation with `jepa_weight=0.3`, `lr=5e-5`, 2 epochs.

The headline comparison is JEPA continuation vs untouched released dSVA. The matched continuation control is useful for diagnosing whether JEPA is helping beyond a poor continuation recipe, but it is not the primary baseline.


## Setup


In [1]:
import os
import torch
from pathlib import Path

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'A local CUDA GPU is required for this experiment.'

cwd = Path.cwd()
candidates = [cwd, cwd.parent]
REPO_ROOT = next(
    (path for path in candidates if (path / 'scripts' / 'sweep_dsva_jepa_finetune.py').exists()),
    None,
)
assert REPO_ROOT is not None, f'Could not find repo root from {cwd}'
print('Repo root:', REPO_ROOT)


CUDA available: True
GPU: NVIDIA GeForce RTX 2060 SUPER
Repo root: d:\Florent\Desktop\jepa-transfer-attacks


## Configure ImageNet Paths


In [2]:
# Set these environment variables before launching Jupyter, or edit the paths below.
# Expected layout: ImageNet-style class folders. Numeric folders work directly; WNID folders need CLASS_MAP.

def first_existing(*paths):
    for path in paths:
        if path and Path(path).exists():
            return Path(path)
    return None

PHASE14_OUTPUT_DIR = REPO_ROOT / 'results' / 'phase14_imagenet_validation'
PHASE14_ANALYSIS_DIR = REPO_ROOT / 'results' / 'phase14_analysis'
PHASE14_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PHASE14_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

IMAGENET_TRAIN_ROOT = first_existing(
    os.environ.get('IMAGENET_TRAIN_ROOT'),
    REPO_ROOT / 'imagenet_phase14' / 'train',
    REPO_ROOT / 'imagenet' / 'train',
    REPO_ROOT / 'ILSVRC2012' / 'train',
)
IMAGENET_VAL_ROOT = first_existing(
    os.environ.get('IMAGENET_VAL_ROOT'),
    REPO_ROOT / 'imagenet_phase14' / 'val',
    REPO_ROOT / 'imagenet' / 'val',
    REPO_ROOT / 'ILSVRC2012' / 'val',
)
CLASS_MAP = first_existing(
    os.environ.get('IMAGENET_CLASS_MAP'),
    REPO_ROOT / 'imagenet_phase14' / 'imagenet_class_map.json',
    REPO_ROOT / 'configs' / 'imagenet_class_map.json',
)
DSVA_CHECKPOINT = REPO_ROOT / 'external' / 'models' / 'dSVA' / 'model.pth'

assert IMAGENET_TRAIN_ROOT is not None, 'Set IMAGENET_TRAIN_ROOT or edit this cell.'
assert IMAGENET_VAL_ROOT is not None, 'Set IMAGENET_VAL_ROOT or edit this cell.'
assert DSVA_CHECKPOINT.exists(), f'Missing dSVA checkpoint: {DSVA_CHECKPOINT}'

print('ImageNet train root:', IMAGENET_TRAIN_ROOT)
print('ImageNet val root:', IMAGENET_VAL_ROOT)
print('Class map:', CLASS_MAP if CLASS_MAP else 'none yet')
print('dSVA checkpoint:', DSVA_CHECKPOINT)


ImageNet train root: d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\train
ImageNet val root: d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\val
Class map: d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\imagenet_class_map.json
dSVA checkpoint: d:\Florent\Desktop\jepa-transfer-attacks\external\models\dSVA\model.pth


## Auto-Create ImageNet Class Map


In [3]:
import json

if CLASS_MAP is None:
    try:
        from torchvision.datasets import ImageNet
        imagenet_root = IMAGENET_TRAIN_ROOT.parent
        metadata = ImageNet(str(imagenet_root), split='val')
        generated_map = PHASE14_ANALYSIS_DIR / 'imagenet_class_map_from_torchvision.json'
        generated_map.parent.mkdir(parents=True, exist_ok=True)
        generated_map.write_text(json.dumps(metadata.wnid_to_idx, indent=2, sort_keys=True), encoding='utf-8')
        CLASS_MAP = generated_map
        print('Generated class map from torchvision metadata:', CLASS_MAP)
    except Exception as exc:
        print('Could not auto-generate class map from torchvision ImageNet metadata.')
        print('If your ImageNet folders are WNIDs, set IMAGENET_CLASS_MAP to a JSON mapping WNID -> class index.')
        print('If folders are numeric 0..999, no class map is needed.')
        print('Details:', repr(exc))
else:
    print('Using class map:', CLASS_MAP)


Using class map: d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\imagenet_class_map.json


## Experiment Configuration


In [4]:
SEEDS = [0, 1, 2]
TRAIN_LIMIT = 1000
EVAL_LIMIT = 50000
EPSILON = '0.06274509803921569'
LR = '0.00005'
EPOCHS = 2
BEST_CONFIG = '0.3:0.00005'
OUTPUT_MODE = 'scaled-delta'
VICTIMS = ['resnet50', 'convnext_tiny', 'vit_b_16', 'efficientnet_b0', 'swin_t']

TRAIN_PERF_FLAGS = ['--compile', '--compile-mode', 'reduce-overhead']
EVAL_PERF_FLAGS = ['--eval-amp', *TRAIN_PERF_FLAGS]
CLASS_MAP_FLAGS = ['--class-map', str(CLASS_MAP)] if CLASS_MAP else []

print('Output dir:', PHASE14_OUTPUT_DIR)
print('Analysis dir:', PHASE14_ANALYSIS_DIR)
print('Train limit:', TRAIN_LIMIT)
print('Eval limit:', EVAL_LIMIT)
print('Victims:', VICTIMS)
print('Best config:', BEST_CONFIG)
print('Class-map flags:', CLASS_MAP_FLAGS)


Output dir: d:\Florent\Desktop\jepa-transfer-attacks\results\phase14_imagenet_validation
Analysis dir: d:\Florent\Desktop\jepa-transfer-attacks\results\phase14_analysis
Train limit: 1000
Eval limit: 50000
Victims: ['resnet50', 'convnext_tiny', 'vit_b_16', 'efficientnet_b0', 'swin_t']
Best config: 0.3:0.00005
Class-map flags: ['--class-map', 'd:\\Florent\\Desktop\\jepa-transfer-attacks\\imagenet_phase14\\imagenet_class_map.json']


## Helpers


In [5]:
import subprocess, sys
from pathlib import Path

def run_checked(cmd):
    print('\n' + ' '.join(map(str, cmd)), flush=True)
    log_dir = REPO_ROOT / 'results' / 'phase14_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (Path(str(cmd[1])).stem + '_' + str(len(list(log_dir.glob('*.log')))) + '.log')
    with log_path.open('w', encoding='utf-8', errors='replace') as handle:
        result = subprocess.run(
            cmd,
            cwd=REPO_ROOT,
            text=True,
            stdout=handle,
            stderr=subprocess.STDOUT,
        )
    text = log_path.read_text(encoding='utf-8', errors='replace')
    print(text[-12000:])
    print('Log:', log_path)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


## Smoke Test


In [6]:
smoke_csv = REPO_ROOT / 'results' / 'phase14_smoke_released_dsva_imagenet.csv'
if smoke_csv.exists():
    print('Skipping existing smoke CSV:', smoke_csv)
else:
    run_checked([
        sys.executable,
        'scripts/run_dsva_checkpoint_attack.py',
        '--data-root', str(IMAGENET_VAL_ROOT),
        '--checkpoint', str(DSVA_CHECKPOINT),
        '--output-mode', 'adv',
        '--limit', '32',
        '--batch-size', '8',
        '--epsilon', EPSILON,
        '--victims', *VICTIMS,
        '--device', 'cuda',
        '--output-csv', str(smoke_csv),
        '--amp',
        *CLASS_MAP_FLAGS,
    ])



c:\Users\Florent\anaconda3\python.exe scripts/run_dsva_checkpoint_attack.py --data-root d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\val --checkpoint d:\Florent\Desktop\jepa-transfer-attacks\external\models\dSVA\model.pth --output-mode adv --limit 32 --batch-size 8 --epsilon 0.06274509803921569 --victims resnet50 convnext_tiny vit_b_16 efficientnet_b0 swin_t --device cuda --output-csv d:\Florent\Desktop\jepa-transfer-attacks\results\phase14_smoke_released_dsva_imagenet.csv --amp --class-map d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\imagenet_class_map.json

dSVA checkpoint transfer: 100%|██████████| 4/4 [00:20<00:00,  5.07s/it]
| model | n | clean_acc | adv_acc | acc_drop | attack_success_rate |
| --- | ---: | ---: | ---: | ---: | ---: |
| resnet50 | 32 | 96.88% | 50.00% | 46.88% | 48.39% |
| convnext_tiny | 32 | 93.75% | 65.62% | 28.12% | 30.00% |
| vit_b_16 | 32 | 96.88% | 15.62% | 81.25% | 83.87% |
| efficientnet_b0 | 32 | 90.62% | 6.25% | 84.38% | 93.10

## Untouched Released dSVA ImageNet Baseline


In [7]:
untouched_csv = PHASE14_ANALYSIS_DIR / 'official_dsva_imagenet_eval.csv'
if untouched_csv.exists():
    print('Skipping existing untouched ImageNet baseline:', untouched_csv)
else:
    run_checked([
        sys.executable,
        'scripts/run_dsva_checkpoint_attack.py',
        '--data-root', str(IMAGENET_VAL_ROOT),
        '--checkpoint', str(DSVA_CHECKPOINT),
        '--output-mode', 'adv',
        '--limit', str(EVAL_LIMIT),
        '--batch-size', '8',
        '--epsilon', EPSILON,
        '--victims', *VICTIMS,
        '--device', 'cuda',
        '--output-csv', str(untouched_csv),
        '--amp',
        *CLASS_MAP_FLAGS,
    ])



c:\Users\Florent\anaconda3\python.exe scripts/run_dsva_checkpoint_attack.py --data-root d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\val --checkpoint d:\Florent\Desktop\jepa-transfer-attacks\external\models\dSVA\model.pth --output-mode adv --limit 50000 --batch-size 8 --epsilon 0.06274509803921569 --victims resnet50 convnext_tiny vit_b_16 efficientnet_b0 swin_t --device cuda --output-csv d:\Florent\Desktop\jepa-transfer-attacks\results\phase14_analysis\official_dsva_imagenet_eval.csv --amp --class-map d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\imagenet_class_map.json
dSVA checkpoint transfer: 100%|██████████| 6250/6250 [1:02:37<00:00,  1.66it/s]
| model | n | clean_acc | adv_acc | acc_drop | attack_success_rate |
| --- | ---: | ---: | ---: | ---: | ---: |
| resnet50 | 50000 | 80.15% | 27.83% | 52.32% | 66.69% |
| convnext_tiny | 50000 | 81.86% | 39.90% | 41.96% | 52.85% |
| vit_b_16 | 50000 | 80.89% | 16.34% | 64.55% | 80.46% |
| efficientnet_b0 | 50000 | 7

In [8]:
import pandas as pd

untouched = pd.read_csv(untouched_csv)
untouched_mean = untouched['attack_success_rate'].mean()
print('Untouched ImageNet mean:', f'{100 * untouched_mean:.2f}%')
untouched


Untouched ImageNet mean: 68.92%


,model,n,clean_acc,adv_acc,acc_drop,attack_success_rate
0,resnet50,50000,0.80148,0.27826,0.52322,0.666891
1,convnext_tiny,50000,0.81860,0.39900,0.41960,0.528537
2,vit_b_16,50000,0.80892,0.16340,0.64552,0.804604
3,efficientnet_b0,50000,0.77688,0.04896,0.72792,0.940480
4,swin_t,50000,0.81332,0.41514,0.39818,0.505533


## Best JEPA Continuation vs Matched Control


In [9]:
for seed in SEEDS:
    seed_dir = PHASE14_OUTPUT_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        'scripts/sweep_dsva_jepa_finetune.py',
        '--train-root', str(IMAGENET_TRAIN_ROOT),
        '--val-root', str(IMAGENET_VAL_ROOT),
        '--init-checkpoint', str(DSVA_CHECKPOINT),
        '--output-dir', str(seed_dir),
        '--run-prefix', f'phase14_seed{seed}_imagenet_jw0p3_ep{EPOCHS}',
        '--configs', BEST_CONFIG,
        '--limit', str(TRAIN_LIMIT),
        '--eval-limit', str(EVAL_LIMIT),
        '--epochs', str(EPOCHS),
        '--batch-size', '1',
        '--eval-batch-size', '8',
        '--grad-accum-steps', '8',
        '--epsilon', EPSILON,
        '--output-mode', OUTPUT_MODE,
        '--victims', *VICTIMS,
        '--device', 'cuda',
        '--seed', str(seed),
        '--normalize-loss-weights',
        '--skip-existing',
        *EVAL_PERF_FLAGS,
        *CLASS_MAP_FLAGS,
    ]
    run_checked(cmd)



c:\Users\Florent\anaconda3\python.exe scripts/sweep_dsva_jepa_finetune.py --train-root d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\train --val-root d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\val --init-checkpoint d:\Florent\Desktop\jepa-transfer-attacks\external\models\dSVA\model.pth --output-dir d:\Florent\Desktop\jepa-transfer-attacks\results\phase14_imagenet_validation\seed_0 --run-prefix phase14_seed0_imagenet_jw0p3_ep2 --configs 0.3:0.00005 --limit 1000 --eval-limit 50000 --epochs 2 --batch-size 1 --eval-batch-size 8 --grad-accum-steps 8 --epsilon 0.06274509803921569 --output-mode scaled-delta --victims resnet50 convnext_tiny vit_b_16 efficientnet_b0 swin_t --device cuda --seed 0 --normalize-loss-weights --skip-existing --eval-amp --compile --compile-mode reduce-overhead --class-map d:\Florent\Desktop\jepa-transfer-attacks\imagenet_phase14\imagenet_class_map.json
dSVA checkpoint transfer: 100%|██████████| 6250/6250 [41:24<00:00,  2.52it/s]
| model | n

## Aggregate and Compare


In [10]:
run_checked([
    sys.executable,
    'scripts/analyze_phase11_scale_results.py',
    '--scale-root', 'results/phase14_imagenet_validation',
    '--output-detail-csv', 'results/phase14_analysis/imagenet_detail.csv',
    '--output-aggregate-csv', 'results/phase14_analysis/imagenet_aggregate.csv',
])

aggregate = pd.read_csv(PHASE14_ANALYSIS_DIR / 'imagenet_aggregate.csv')
aggregate



c:\Users\Florent\anaconda3\python.exe scripts/analyze_phase11_scale_results.py --scale-root results/phase14_imagenet_validation --output-detail-csv results/phase14_analysis/imagenet_detail.csv --output-aggregate-csv results/phase14_analysis/imagenet_aggregate.csv
Wrote detail CSV: results\phase14_analysis\imagenet_detail.csv
Wrote aggregate CSV: results\phase14_analysis\imagenet_aggregate.csv

Log: d:\Florent\Desktop\jepa-transfer-attacks\results\phase14_logs\analyze_phase11_scale_results_5.log


,objectives,run_type,jepa_weight,lr,epochs,train_limit,num_seeds,mean_transfer_success,std_transfer_success,resnet50_mean,resnet50_std,convnext_tiny_mean,convnext_tiny_std,vit_b_16_mean,vit_b_16_std,efficientnet_b0_mean,efficientnet_b0_std,swin_t_mean,swin_t_std
0,dino_mae,control,0.0,0.00005,2,1000,3,0.707670,0.012077,0.679377,0.026652,0.521895,0.018520,0.839663,0.000858,0.950318,0.005055,0.547099,0.011110
1,dino_mae_jepa,jepa,0.3,0.00005,2,1000,3,0.699803,0.010318,0.671791,0.025799,0.504263,0.017288,0.831899,0.006750,0.952790,0.005798,0.538271,0.007425


In [11]:
rows = aggregate.copy()
rows['mean_transfer_success'] = rows['mean_transfer_success'].astype(float)
control = rows[rows['run_type'] == 'control'].iloc[0]
jepa = rows[rows['run_type'] == 'jepa'].iloc[0].copy()

summary = {
    'untouched_mean': untouched_mean,
    'control_mean': float(control['mean_transfer_success']),
    'jepa_mean': float(jepa['mean_transfer_success']),
    'gain_vs_untouched': float(jepa['mean_transfer_success']) - untouched_mean,
    'gain_vs_control': float(jepa['mean_transfer_success']) - float(control['mean_transfer_success']),
}
print({key: f'{100 * value:.2f}%' if 'mean' in key else f'{100 * value:+.2f} pts' for key, value in summary.items()})

victim_baseline = untouched.set_index('model')['attack_success_rate'].to_dict()
for victim in VICTIMS:
    column = f'{victim}_mean'
    if column in jepa and victim in victim_baseline:
        print(victim, f"{100 * (float(jepa[column]) - victim_baseline[victim]):+.2f} pts vs untouched")


{'untouched_mean': '68.92%', 'control_mean': '70.77%', 'jepa_mean': '69.98%', 'gain_vs_untouched': '+1.06 pts', 'gain_vs_control': '-0.79 pts'}
resnet50 +0.49 pts vs untouched
convnext_tiny -2.43 pts vs untouched
vit_b_16 +2.73 pts vs untouched
efficientnet_b0 +1.23 pts vs untouched
swin_t +3.27 pts vs untouched


## Decision Read


In [12]:
gain = summary['gain_vs_untouched']
if gain >= 0.01:
    print('Decision: full ImageNet validation supports a workshop/short-paper claim.')
elif gain > 0:
    print('Decision: positive but modest; report carefully or add per-image complementarity before claiming scale.')
else:
    print('Decision: do not claim ImageNet-scale improvement; keep the result as an Imagenette/domain-adaptation finding.')


Decision: full ImageNet validation supports a workshop/short-paper claim.


## Package CSV Outputs


In [13]:
import tarfile

archive_path = REPO_ROOT / 'results' / 'phase14_csv_artifacts.tar.gz'
paths = list(PHASE14_OUTPUT_DIR.rglob('*.csv'))
paths += list(PHASE14_ANALYSIS_DIR.rglob('*.csv'))
if smoke_csv.exists():
    paths.append(smoke_csv)

with tarfile.open(archive_path, 'w:gz') as tar:
    for path in paths:
        tar.add(path, arcname=path.relative_to(REPO_ROOT))
print('Archive:', archive_path)


Archive: d:\Florent\Desktop\jepa-transfer-attacks\results\phase14_csv_artifacts.tar.gz
